# Selección del mejor modelo

Este notebook tiene un proposito simple, elegitr el mejor modelo entre los 4 modelos de demanda y pasarlo a un pipeline de spark para usarlo en inferencia lo más fácil posible.

In [1]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import joblib

root = Path.cwd().parent.parent
sys.path.append(str(root))

from minio_utils import MinioSparkClient


from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
    SQLTransformer,
    Imputer,
)
from pyspark.sql.types import FloatType
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml import Transformer
from pyspark.ml.util import DefaultParamsWritable, DefaultParamsReadable
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
import math
from torch.utils.data import IterableDataset, DataLoader
import pyarrow.parquet as pq

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import TiDEModel, TFTModel
from darts.dataprocessing.transformers import MissingValuesFiller
from darts.utils.likelihood_models import QuantileRegression
from sklearn.preprocessing import RobustScaler

from clearml import Task
from tqdm.auto import tqdm

from setup import setenv

setenv()

os.environ["JAVA_TOOL_OPTIONS"] = (
    "-Dio.netty.tryReflectionSetAccessible=true --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED"
)
spark = MinioSparkClient(
    endpoint=os.getenv("MINIO_ENDPOINT", "")
    .replace("http://", "")
    .replace("https://", ""),
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    bucket_name="pd2",
    base_dir="cityenjoyer",
    memory=16,
    heapsize=8,
    num_part=2000,
    inference=True,
    verbose=True,
)
spark.connect()

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
Picked up JAVA_TOOL_OPTIONS: -Dio.netty.tryReflectionSetAccessible=true --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: -Dio.netty.tryReflectionSetAccessible=true --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED
26/04/09 17:54:35 WARN Utils: Your hostname, danpanto-OMEN-Gaming-Laptop-16-ap0xxx resolves to a loopback address: 127.0.1.1; using 192.168.7.79 instead (on interface eno1)
26/04/09 17:54:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://mmlspark.azureedge.net/maven added as a remote repository with the name: repo-1
Ivy Default

:: loading settings :: url = jar:file:/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.microsoft.azure#synapseml_2.12;1.1.2 in central
	found com.microsoft.azure#synapseml-core_2.12;1.1.2 in central
	found org.apache.spark#spark-avro_2.12;3.5.0 in central
	found org.tukaani#xz;1.9 in central
	found commons-lang#commons-lang;2.6 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central
	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central

In [2]:
df = spark.read_parquet("prepared_for_model/20260405_154246_agg.parquet")

df.show()

+--------+------------+-------------------+------+------------+----------+---------+----------+----+---+--------------------+--------------------+-------------------+--------------------+
|VendorID|PULocationID|          timestamp|demand|avg_distance|avg_amount| Latitude| Longitude|hour|dow|            hour_sin|            hour_cos|            dow_sin|             dow_cos|
+--------+------------+-------------------+------+------------+----------+---------+----------+----+---+--------------------+--------------------+-------------------+--------------------+
|       1|         166|2026-02-01 07:00:00|     2|      3113.5|    1511.0|40.809456| -73.96176|   7|  1|  0.9659258262890683|-0.25881904510252063| 0.7818314824680298|  0.6234898018587336|
|       1|          74|2026-02-02 10:00:00|    30|   3326.5334| 2484.7666| 40.80117| -73.93735|  10|  2|  0.5000000000000003| -0.8660254037844385| 0.9749279121818236|-0.22252093395631434|
|       1|          74|2026-02-01 17:00:00|    24|   3352.29

Estos datos corresponden a los datos hasta la fecha que quedaban, diciembre enero y febrero, además son los unicos necesarios para inferir nuevos datos.

Antes de nada creemos una función que evalue el test dado un dataframe.

In [3]:
import pyspark.sql.functions as F


def evaluar_spark(df, df_pred):
    """
    Evalúa modelos a 1-Step (1 hora vista) para comparar modelos simples con Darts.
    """

    print("Calculando la siguiente hora...")

    # 4. UNIR PREDICCIONES CON LA REALIDAD
    df_eval = df.select("VendorID", "PULocationID", "timestamp", "demand").join(
        df_pred, on=["VendorID", "PULocationID", "timestamp"], how="inner"
    )
    # 5. MÉTRICAS COMPARATIVAS
    df_metricas = df_eval.select(
        F.mean(F.abs(F.col("demand") - F.col("prediction_real"))).alias("MAE"),
        F.sqrt(F.mean(F.pow(F.col("demand") - F.col("prediction_real"), 2))).alias(
            "RMSE"
        ),
        F.median(F.abs(F.col("demand") - F.col("prediction_real"))).alias("MedAE"),
    )

    df_metricas.show()

Veamos como predice estos datos la LSTM que hicimos al principio

In [4]:
class ExTLSTM(nn.Module):
    def __init__(self, vendor_dim, window_size=5, feature_dim=3):
        super(ExTLSTM, self).__init__()
        self.vendor_dim = vendor_dim
        self.window_size = window_size
        self.feature_dim = feature_dim

        self.vendor_fc = nn.Linear(vendor_dim, 16)
        self.centroid_fc = nn.Linear(2, 16)
        self.time_fc = nn.Linear(4, 16)

        self.lstm = nn.LSTM(
            input_size=feature_dim, hidden_size=64, num_layers=1, batch_first=True
        )

        # Entrada final: 16 (vendor) + 16 (lat/lon) + 16 (day/week)  + (64 hidden states)
        combined_dim = 16 + 16 + 16 + 64

        self.fc_final = nn.Sequential(
            nn.Linear(combined_dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1)
        )

    def forward(self, x):
        idx_v = self.vendor_dim
        idx_c = idx_v + 2
        idx_t = idx_c + 4

        vendor_data = x[:, :idx_v]
        centroid_data = x[:, idx_v:idx_c]
        time_data = x[:, idx_c:idx_t]

        temporal_data = x[:, idx_t:].view(-1, self.window_size, self.feature_dim)

        v_out = torch.relu(self.vendor_fc(vendor_data))
        c_out = torch.relu(self.centroid_fc(centroid_data))
        t_out = torch.relu(self.time_fc(time_data))

        out, _ = self.lstm(temporal_data)

        lstm_out = torch.mean(out, dim=1)

        # Concatenamos todo
        combined = torch.cat((v_out, c_out, t_out, lstm_out), dim=1)
        return self.fc_final(combined)


class LSTMPredictor(
    Transformer, HasInputCol, HasOutputCol, DefaultParamsWritable, DefaultParamsReadable
):
    def __init__(self, model_state=None, inputCol=None, outputCol=None):
        super(LSTMPredictor, self).__init__()
        self.model_state = model_state
        self.setParams(inputCol=inputCol, outputCol=outputCol)

    def setParams(self, inputCol=None, outputCol=None):
        return self._set(inputCol=inputCol, outputCol=outputCol)

    def _transform(self, dataset):
        state = self.model_state

        # Usamos una UDF estándar de Python (no de Pandas/Arrow)
        def predict_single(features):
            device = torch.device(
                "cpu"
            )  # Usamos CPU para máxima estabilidad en el Pipeline
            model = ExTLSTM(vendor_dim=4, window_size=5, feature_dim=3)
            model.load_state_dict(state)
            model.eval()

            # Convertimos el Vector de Spark a Tensor
            feat_array = np.array(features.toArray()).astype(np.float32)
            feat_tensor = torch.from_numpy(feat_array).unsqueeze(0)

            with torch.no_grad():
                pred = model(feat_tensor).item()
            return float(pred)

        predict_udf = F.udf(predict_single, FloatType())

        return dataset.withColumn(
            self.getOutputCol(), predict_udf(F.col(self.getInputCol()))
        )


# La siguiente expession es el equivalente al rolling window
def get_pipeline_completo(label):
    """
    Crea y devuelve un pipeline para una variable objetivo label
    """
    sql_logic = f"""
    SELECT *,
        -- Lags de DEMANDA, DISTANCIA y MONTO
        COALESCE(LAG(demand_sc, 5) OVER (p), 0.0) as d_t5, COALESCE(LAG(demand_sc, 4) OVER (p), 0.0) as d_t4, 
        COALESCE(LAG(demand_sc, 3) OVER (p), 0.0) as d_t3, COALESCE(LAG(demand_sc, 2) OVER (p), 0.0) as d_t2, COALESCE(LAG(demand_sc, 1) OVER (p), 0.0) as d_t1,
     
        COALESCE(LAG(avg_distance_sc, 5) OVER (p), 0.0) as dist_t5, COALESCE(LAG(avg_distance_sc, 4) OVER (p), 0.0) as dist_t4, 
        COALESCE(LAG(avg_distance_sc, 3) OVER (p), 0.0) as dist_t3, COALESCE(LAG(avg_distance_sc, 2) OVER (p), 0.0) as dist_t2, COALESCE(LAG(avg_distance_sc, 1) OVER (p), 0.0) as dist_t1,

        COALESCE(LAG(avg_amount_sc, 5) OVER (p), 0.0) as amt_t5, COALESCE(LAG(avg_amount_sc, 4) OVER (p), 0.0) as amt_t4, 
        COALESCE(LAG(avg_amount_sc, 3) OVER (p), 0.0) as amt_t3, COALESCE(LAG(avg_amount_sc, 2) OVER (p), 0.0) as amt_t2, COALESCE(LAG(avg_amount_sc, 1) OVER (p), 0.0) as amt_t1
    FROM (
        SELECT *,
            COALESCE(AVG({label}) OVER (w), 0) as mean,
            COALESCE(STDDEV({label}) OVER (w), 1) as std,

            COALESCE((demand - AVG(demand) OVER (w)) / COALESCE(NULLIF(STDDEV(demand) OVER (w), 0.0), 1.0), 0.0) as demand_sc,
            COALESCE((avg_distance - AVG(avg_distance) OVER (w)) / COALESCE(NULLIF(STDDEV(avg_distance) OVER (w), 0.0), 1.0), 0.0) as avg_distance_sc,
            COALESCE((avg_amount - AVG(avg_amount) OVER (w)) / COALESCE(NULLIF(STDDEV(avg_amount) OVER (w), 0.0), 1.0), 0.0) as avg_amount_sc
        FROM __THIS__
        WINDOW w AS (PARTITION BY VendorID, PULocationID ORDER BY timestamp ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING)
    ) AS Escalamiento
    WINDOW p AS (PARTITION BY VendorID, PULocationID ORDER BY timestamp)
    """
    window_transformer = SQLTransformer(statement=sql_logic)

    # Hacemos el one-hot
    indexer = StringIndexer(
        inputCol="VendorID", outputCol="VendorIndex", handleInvalid="keep"
    )
    encoder = OneHotEncoder(inputCol="VendorIndex", outputCol="VendorVector")

    # Escalamos las coordenadas
    coord_assembler = VectorAssembler(
        inputCols=["Latitude", "Longitude"], outputCol="coords_vec"
    )
    coord_scaler = StandardScaler(
        inputCol="coords_vec", outputCol="coords_scaled", withMean=True, withStd=True
    )

    # Ensamblado del Vector Final
    # El orden aquí debe coincidir con el despiece del modelo en PyTorch: [Vendor, Centroide, Secuencia]
    temporal_features = []
    for i in range(5, 0, -1):
        temporal_features += [
            f"d_t{i}",
            f"dist_t{i}",
            f"amt_t{i}",
        ]

    current_time_features = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]

    assembler = VectorAssembler(
        inputCols=["VendorVector", "coords_scaled"]
        + current_time_features
        + temporal_features,
        outputCol="features",
    )

    model_weights = torch.load("../best_model.pth")
    lstm_stage = LSTMPredictor(
        model_state=model_weights, inputCol="features", outputCol="prediction"
    )

    logic_denorm = """
    SELECT *,
        (prediction * std) + mean as prediction_real
    FROM __THIS__
    """

    denormalizer = SQLTransformer(statement=logic_denorm)
    # Pipeline completo
    return Pipeline(
        stages=[
            window_transformer,
            indexer,
            encoder,
            coord_assembler,
            coord_scaler,
            assembler,
            lstm_stage,
            denormalizer,
        ]
    )


pipeline_lstm = get_pipeline_completo("demand")
pred = pipeline_lstm.fit(df).transform(df)

In [10]:
pred.show()

+--------+------------+-------------------+------+------------+----------+---------+----------+----+---+--------------------+--------------------+-------------------+--------------------+------------------+------------------+--------------------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+--------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------+-------------+--------------------+--------------------+--------------------+------------+--------------------+
|VendorID|PULocationID|          timestamp|demand|avg_distance|avg_amount| Latitude| Longitude|hour|dow|            hour_sin|            hour_cos|            dow_sin|             dow_cos|              mean|               std|           demand_sc|     avg_distance_sc|       avg_

In [5]:
evaluar_spark(
    df, pred.select("VendorID", "PULocationID", "timestamp", "prediction_real")
)

Calculando la siguiente hora...


+-----------------+-----------------+-----------------+
|              MAE|             RMSE|            MedAE|
+-----------------+-----------------+-----------------+
|8.535918778034109|16.64338243034058|4.141061990428815|
+-----------------+-----------------+-----------------+



Este modelo como veremos es el peor, una muy buena aproximación con un entrenamiento rapido, pero solo predice 1 hora, mientras que el resto de modelos son xcapaces de predecir 24 horas de golpe, lo que los hacen modelos muy potentes. Evaluemos los dos Tide y el TFT 

Para ello, aunque en inferencia simplemente pasemos a local un extracto, pasaremos los datos al formato de entrenamiento, ya que darts da muchos problemas con spark

In [ ]:
# Limpiamos nulos
df_clean = df.na.drop(subset=["demand", "PULocationID", "VendorID", "timestamp"])

# Creamos una columna que sea el ID único de la serie temporal
df_clean = df_clean.withColumn(
    "Series_ID", F.concat_ws("_", F.col("VendorID"), F.col("PULocationID"))
)

# Ahora particionamos por esa nueva columna,
# esto nos permite tener un folder por cada combinación y
# nos permite usar darts con mini datasets que podemos pasar a pandas
df_clean.write.partitionBy("Series_ID").mode("overwrite").parquet(
    "../data/data_test.parquet"
)

print("Datos exportados correctamente por Vendor y Zona.")

Datos exportados correctamente por Vendor y Zona.


In [11]:
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

root = Path.cwd().parent
sys.path.append(str(root))

from minio_utils import MinioSparkClient


from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
    SQLTransformer,
    Imputer,
)
from pyspark.sql.types import FloatType
from pyspark.sql import functions as F

from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
import math
from torch.utils.data import IterableDataset, DataLoader
import pyarrow.parquet as pq

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import TiDEModel
from darts.dataprocessing.transformers import MissingValuesFiller
from darts.utils.likelihood_models import QuantileRegression
from pytorch_lightning.callbacks import EarlyStopping
from sklearn.preprocessing import RobustScaler


from clearml import Task
from tqdm.auto import tqdm
import joblib
import pandas as pd
from darts import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder as SklearnOneHotEncoder
from sklearn.cluster import KMeans


from setup import setenv

setenv()

In [17]:
# importamos lod tres modelos de Darts que hemos entrenado y los trasnformers para invertir las transformaciones de los datos
import json

with open("../darts_models/router_zonas.json", "r") as archivo:
    router_zonas = json.load(archivo)
escalador_distancia = joblib.load("../darts_models/scaler_espacial.pkl")
encoder_vendor = joblib.load("../darts_models/encoder_vendor.pkl")

modelo_high = TiDEModel.load_from_checkpoint(
    model_name="tide_nyc_alta_v5",
    work_dir="../darts_models/",
    best=True,
    weights_only=False,
)
modelo_mid = TiDEModel.load_from_checkpoint(
    model_name="tide_nyc_media_v5",
    work_dir="../darts_models/",
    best=True,
    weights_only=False,
)
modelo_baseline = TiDEModel.load_from_checkpoint(
    model_name="tide_nyc_baseline_v5",
    work_dir="../darts_models/",
    best=True,
    weights_only=False,
)

In [13]:
df_pd = pd.read_parquet("../data/data_test.parquet")

# Es vital que los datos estén ordenados temporalmente dentro de cada grupo
df_pd = df_pd.sort_values(by=["Series_ID", "timestamp"])

fecha_limite = df_pd["timestamp"].max() - pd.DateOffset(years=3)
df_pd = df_pd[df_pd["timestamp"] >= fecha_limite].copy()


# LATITUD Y LONGITUD (Escalado espacial)
# Escalamos las coordenadas entre 0 y 1 en Pandas para que TiDE las digiera perfecto
cols_geo = ["Latitude", "Longitude"]

df_pd[cols_geo] = escalador_distancia.transform(df_pd[cols_geo])
df_pd[cols_geo] = df_pd[cols_geo].astype("float32")

# VARIABLES NUMÉRICAS (Float32)
cols_numericas = ["demand", "avg_distance", "avg_amount"]
df_pd[cols_numericas] = df_pd[cols_numericas].astype("float32")

print("Aplicando One-Hot Encoding a VendorID...")


dummies_array = encoder_vendor.transform(df_pd[["VendorID"]])

nombres_dummies = encoder_vendor.get_feature_names_out(["VendorID"])

df_pd[nombres_dummies] = dummies_array

df_pd["VendorID"] = df_pd["VendorID"].astype(
    "float32"
)  # Mantenemos el original para Darts

columnas_estaticas = list(nombres_dummies) + ["Latitude", "Longitude"]

print(f"Variables estáticas que verá el modelo: {columnas_estaticas}")

df_pd[columnas_estaticas] = df_pd[columnas_estaticas].astype("float32")

print(f"Variables estáticas que verá el modelo: {columnas_estaticas}")


df_pd["segmento_asignado"] = (
    df_pd["Series_ID"]
    .astype(str)
    .apply(lambda id_zona: router_zonas.get(id_zona, "BASELINE"))
)


df_baseline = df_pd[df_pd["segmento_asignado"] == "BASELINE"].copy()
df_medias = df_pd[df_pd["segmento_asignado"] == "MID"].copy()
df_altas = df_pd[df_pd["segmento_asignado"] == "HIGH"].copy()


col_tiempo = "timestamp"
col_grupo = ["PULocationID", "VendorID"]

#######
# Preparam,os un calendario global para que todas las series tengan el mismo rango temporal, incluso si hay huecos
#######

col_tiempo = "timestamp"
col_grupo = ["PULocationID", "VendorID"]

# 1. CREAMOS EL CALENDARIO MAESTRO (Una sola vez para todo el proyecto)
print("Generando Calendario Maestro de Covariables Futuras...")
HORAS_EXTRA = 48
inicio_global = df_pd["timestamp"].min()
fin_global = df_pd["timestamp"].max() + pd.Timedelta(hours=HORAS_EXTRA)

# Rango que cubre absolutamente todo el dataset + 48h de futuro
indice_global = pd.date_range(start=inicio_global, end=fin_global, freq="h")

cov_hora_global = datetime_attribute_timeseries(
    indice_global, attribute="hour", cyclic=True
)
cov_dia_global = datetime_attribute_timeseries(
    indice_global, attribute="dayofweek", cyclic=True
)
calendario_maestro = cov_hora_global.stack(cov_dia_global).astype("float32")


def preparar_dataset_darts(df_segmento, nombre_segmento):
    """Convierte un DataFrame de Pandas en las 3 listas de TimeSeries para TiDE"""
    if df_segmento.empty:
        return [], [], []

    # TARGETS
    targets = TimeSeries.from_group_dataframe(
        df_segmento,
        group_cols=col_grupo,
        time_col=col_tiempo,
        value_cols="demand",
        static_cols=columnas_estaticas,
        fill_missing_dates=True,
        freq="h",
        fillna_value=0,
    )

    # PAST COVARIATES
    past_covs = TimeSeries.from_group_dataframe(
        df_segmento,
        group_cols=col_grupo,
        time_col=col_tiempo,
        value_cols=["avg_distance", "avg_amount"],
        fill_missing_dates=True,
        freq="h",
        fillna_value=0,
    )

    future_covs = [calendario_maestro] * len(targets)

    return targets, past_covs, future_covs


# 1. Datos para el TiDE Ligero
targets_mid, past_mid, future_mid = preparar_dataset_darts(df_medias, "Media Demanda")

# 2. Datos para el TiDE Pesado
targets_high, past_high, future_high = preparar_dataset_darts(df_altas, "Alta Demanda")
# 3. Datos para el TiDE Baseline

targets_baseline, past_baseline, future_baseline = preparar_dataset_darts(
    df_baseline, "Baseline"
)

Aplicando One-Hot Encoding a VendorID...
Variables estáticas que verá el modelo: ['VendorID_1', 'VendorID_2', 'VendorID_3', 'Latitude', 'Longitude']
Variables estáticas que verá el modelo: ['VendorID_1', 'VendorID_2', 'VendorID_3', 'Latitude', 'Longitude']
Generando Calendario Maestro de Covariables Futuras...


In [22]:
from darts.dataprocessing.transformers import InvertibleMapper, Mapper
from darts.dataprocessing.pipeline import Pipeline
import numpy as np

# ==========================================
# PREPARACIÓN DE TEST / INFERENCIA (AUTÓNOMA)
# ==========================================


def preparar_datase(targets, past, future, nombre_segmento):
    """
    Filtra los datos de Test y aplica transformaciones matemáticas puras (Log1p).
    Instancia los pipelines internamente y los devuelve para su uso futuro.
    """
    if not targets:
        return {}, None, None

    print(f"\nPreparando datos de TEST para: {nombre_segmento}...")

    INPUT_CHUNK = 72

    test_tgt = []
    test_pst = []
    test_fut = []

    zonas_descartadas = 0

    for tgt, pst, fut in zip(targets, past, future):
        # ESCUDO: Tamaño mínimo histórico
        if len(tgt) >= INPUT_CHUNK:
            test_tgt.append(tgt)
            test_pst.append(pst)
            test_fut.append(fut)
        else:
            zonas_descartadas += 1

    print(
        f"{nombre_segmento}: {len(test_tgt)} series válidas listas. Descartadas: {zonas_descartadas}"
    )

    if not test_tgt:
        return {}, None, None

    # ==========================================
    # INSTANCIAR Y APLICAR PIPELINES (Al Vuelo)
    # ==========================================
    # Como son funciones matemáticas sin estado, podemos instanciarlas aquí directamente
    pipe_target = Pipeline([InvertibleMapper(fn=np.log1p, inverse_fn=np.expm1)])
    pipe_past = Pipeline([Mapper(fn=np.log1p)])

    dict_test = {
        "targets_real": test_tgt,
        "targets": pipe_target.fit_transform(
            test_tgt
        ),  # Podemos usar fit_transform o transform, da igual porque es estático
        "past": pipe_past.fit_transform(test_pst),
        "future": test_fut,
    }

    # Devolvemos los datos Y los pipelines
    return dict_test, pipe_target, pipe_past


# ==========================================
# 🚀 CÓMO EJECUTARLO AHORA
# ==========================================

test_high, pipe_high, _ = preparar_datase(
    targets_high, past_high, future_high, "ALTA Demanda"
)
test_mid, pipe_target_mid, _ = preparar_datase(
    targets_mid, past_mid, future_mid, "MEDIA Demanda"
)
test_baseline, pipe_baseline, _ = preparar_datase(
    targets_baseline, past_baseline, future_baseline, "BASELINE"
)

print("\n¡Datos de Test procesados y Pipelines listos para invertir resultados!")


Preparando datos de TEST para: ALTA Demanda...
ALTA Demanda: 80 series válidas listas. Descartadas: 0

Preparando datos de TEST para: MEDIA Demanda...
MEDIA Demanda: 220 series válidas listas. Descartadas: 0

Preparando datos de TEST para: BASELINE...
BASELINE: 711 series válidas listas. Descartadas: 7

¡Datos de Test procesados y Pipelines listos para invertir resultados!


Pese a que parezca que el error es mayor, la mediana es de 2, lo que significa que en unos puntos especificos habia picos.

In [23]:
from darts.metrics import mae
import numpy as np

VENTANA_VAL = 168
print("📊 Calculando MAE de Validación y MAE Global...\n")

segmentos = [
    ("BASELINE", modelo_baseline, test_baseline),
    ("MEDIA Demanda", modelo_mid, test_mid),
    ("ALTA Demanda", modelo_high, test_high),
]

# El "saco" donde guardaremos los errores de todas las zonas juntas
errores_globales = []

for nombre, modelo, val_data in segmentos:
    print(f"⏳ Evaluando {nombre}...")

    preds_tf_hist = modelo.historical_forecasts(
        series=val_data["targets"],
        past_covariates=val_data["past"],
        future_covariates=val_data["future"],
        start=72,
        forecast_horizon=24,
        stride=24,
        retrain=False,
        last_points_only=False,
        num_samples=200,
    )

    val_real = pipe_high.inverse_transform(val_data["targets"])
    errores_zona = []

    for i, lista_dias_zona in enumerate(preds_tf_hist):
        errores_dias = []  # Guardaremos el error de cada uno de los 7 días

        # 🔥 EL TRUCO: Evaluamos cada bloque de 24h por separado (Cero .append)
        for dia_pred_tf in lista_dias_zona:
            # Invertimos el logaritmo solo de este día
            dia_pred_real = pipe_high.inverse_transform(dia_pred_tf)
            pred_p50 = dia_pred_real.quantile(0.50)

            # Buscamos la realidad exacta para este día
            actual_real = val_real[i].slice_intersect(pred_p50)

            # Calculamos el error de este día concreto
            error_dia = mae(actual_real, pred_p50)
            errores_dias.append(error_dia)

        # La media de los 7 días es el MAE total de esta zona
        mae_zona_total = np.mean(errores_dias)

        errores_zona.append(mae_zona_total)
        errores_globales.append(mae_zona_total)

    mae_promedio = np.mean(errores_zona)
    print(f"   🎯 MAE [{nombre}]: {mae_promedio:.2f} viajes/hora")

# ==========================================
# 🌍 CÁLCULO DEL MAE GLOBAL
# ==========================================
mae_total_real = np.mean(errores_globales)

print("\n" + "=" * 40)
print(f"🏆 MAE GLOBAL DEL SISTEMA: {mae_total_real:.2f} viajes/hora")
print("=" * 40)

📊 Calculando MAE de Validación y MAE Global...

⏳ Evaluando BASELINE...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 5070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/danpanto/Desktop/C-ity-enjoyers/.v

   🎯 MAE [BASELINE]: 2.11 viajes/hora
⏳ Evaluando MEDIA Demanda...


/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


   🎯 MAE [MEDIA Demanda]: 12.98 viajes/hora
⏳ Evaluando ALTA Demanda...


/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/danpanto/Desktop/C-ity-enjoyers/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


   🎯 MAE [ALTA Demanda]: 28.28 viajes/hora

🏆 MAE GLOBAL DEL SISTEMA: 6.55 viajes/hora


Es mejor este último, tengamos en cuenta que use el modelo sin filtrar 0, que es peor, asi que podrían inclusive mejorar los resultados más